# Embedding Your Own RNA-seq Data with the Gen3 Inference Service

In this notebook, we'll play the role of a researcher bringing their own RNA-seq data to the Genomic AI Commons (GAC):
* Our "own" data will be raw gene expression files for 20 diffuse large B-cell lymphoma (DLBCL) cases from the NCI Center for Cancer Research DLBCL cohort (NCICCR-DLBCL, [Schmitz et al. 2018](https://www.nejm.org/doi/full/10.1056/NEJMoa1801445)), as downloaded from the GDC.
* We'll send each file to the Gen3 Inference Service, which embeds it with the `ctds-rnaseq` model, our in-house RNA-seq embedding model.
* We'll use the Gen3 Embedding Service's similarity search to find each sample's nearest neighbors across all TCGA cancer types, and see whether they come from TCGA's own DLBCL cohort (TCGA-DLBC).
* Finally, we'll visualize our samples alongside every TCGA sample with UMAP.

**Note:** the inference service is only reachable from within the GAC.

## Load the input data

Our input is a zip of 20 RNA-seq gene count files from 20 NCICCR-DLBCL cases, plus a metadata table with one row per file. Since this is our own data, we already know its clinical characteristics. These 20 cases were picked from the cohort's 481 files as those whose embeddings sit closest to TCGA-DLBC (see `scripts/build_rnaseq_inference_data.py`).

In [1]:
import json
import zipfile

import numpy as np
import pandas as pd

INPUT_ZIP = "rnaseq-inference-data/nciccr-dlbcl-rnaseq.zip"
INPUT_METADATA = "rnaseq-inference-data/nciccr-dlbcl-metadata.csv"

metadata = pd.read_csv(INPUT_METADATA)
metadata

,file_name,case_id,sample_ids,project_id,primary_site,disease_type,primary_diagnosis,ann_arbor_clinical_stage,age_in_years,sex_at_birth,race,ethnicity,vital_status
0,e9521f2d-64db-4138-b481-fef845196ed7.rna_seq.a...,DLBCL10451,DLBCL10451-sample,NCICCR-DLBCL,Lymph nodes,Mature B-Cell Lymphomas,"Diffuse large B-cell lymphoma, NOS",Stage II,73.0,male,not reported,not reported,Dead
1,2db8100a-3d40-4003-b5af-2d98b14aac54.rna_seq.a...,DLBCL10508,DLBCL10508-sample,NCICCR-DLBCL,Lymph nodes,Mature B-Cell Lymphomas,"Diffuse large B-cell lymphoma, NOS",Stage IV,64.0,male,not reported,not reported,Alive
2,da59dc13-382a-4152-886b-cfce849a89b0.rna_seq.a...,DLBCL10520,DLBCL10520-sample,NCICCR-DLBCL,Lymph nodes,Mature B-Cell Lymphomas,"Diffuse large B-cell lymphoma, NOS",Stage III,76.0,male,not reported,not reported,Dead
3,1527da57-0042-4dbe-8888-ec5a02524c2e.rna_seq.a...,DLBCL10525,DLBCL10525-sample,NCICCR-DLBCL,Lymph nodes,Mature B-Cell Lymphomas,"Diffuse large B-cell lymphoma, NOS",Stage II,65.0,male,not reported,not reported,Alive
4,083522b7-1005-41cc-b30e-f54321eec7fb.rna_seq.a...,DLBCL10861,DLBCL10861-sample,NCICCR-DLBCL,Lymph nodes,Mature B-Cell Lymphomas,"Diffuse large B-cell lymphoma, NOS",Stage IV,18.0,female,not reported,not reported,Alive
5,3081f123-3a36-457d-8285-97fdbd5b6d76.rna_seq.a...,DLBCL10877,DLBCL10877-sample,NCICCR-DLBCL,Lymph nodes,Mature B-Cell Lymphomas,"Diffuse large B-cell lymphoma, NOS",Stage IV,76.0,female,not reported,not reported,Alive
6,dacb4e9b-59ed-482d-b2cd-0cd1903c4eb1.rna_seq.a...,DLBCL10880,DLBCL10880-sample,NCICCR-DLBCL,Lymph nodes,Mature B-Cell Lymphomas,"Diffuse large B-cell lymphoma, NOS",Stage III,72.0,female,not reported,not reported,Alive
7,0faeb4ab-fe8a-4fc4-a8e2-663fe547d17e.rna_seq.a...,DLBCL10887,DLBCL10887-sample,NCICCR-DLBCL,Lymph nodes,Mature B-Cell Lymphomas,"Diffuse large B-cell lymphoma, NOS",Stage IV,62.0,female,not reported,not reported,Alive
8,7b0d6de4-fc35-43f2-b25f-2f17f3f23f78.rna_seq.a...,DLBCL10931,DLBCL10931-sample,NCICCR-DLBCL,Lymph nodes,Mature B-Cell Lymphomas,"Diffuse large B-cell lymphoma, NOS",Stage IV,52.0,female,not reported,not reported,Alive
9,8e1ab1e8-ae2d-4623-b98d-0dec4ea0ffb9.rna_seq.a...,DLBCL10965,DLBCL10965-sample,NCICCR-DLBCL,Lymph nodes,Mature B-Cell Lymphomas,"Diffuse large B-cell lymphoma, NOS",Stage III,60.0,male,not reported,not reported,Alive


In [2]:
# each input file is a GDC STAR gene counts TSV; the model reads its per-gene TPM values
with zipfile.ZipFile(INPUT_ZIP) as zf:
    print(f"{len(zf.namelist())} files in {INPUT_ZIP}")
    with zf.open(metadata["file_name"][0]) as f:
        display(pd.read_csv(f, sep="\t", comment="#").head(10))

20 files in rnaseq-inference-data/nciccr-dlbcl-rnaseq.zip


,gene_id,gene_name,gene_type,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
0,N_unmapped,NaN,NaN,13214994,13214994,13214994,NaN,NaN,NaN
1,N_multimapping,NaN,NaN,27825741,27825741,27825741,NaN,NaN,NaN
2,N_noFeature,NaN,NaN,2991850,30273042,30314546,NaN,NaN,NaN
3,N_ambiguous,NaN,NaN,7214769,1934158,1928898,NaN,NaN,NaN
4,ENSG00000000003.15,TSPAN6,protein_coding,164,93,71,2.0344,0.7417,0.9237
5,ENSG00000000005.6,TNMD,protein_coding,0,0,0,0.0000,0.0000,0.0000
6,ENSG00000000419.13,DPM1,protein_coding,2401,1223,1178,111.9286,40.8053,50.8200
7,ENSG00000000457.14,SCYL3,protein_coding,561,634,677,4.5861,1.6719,2.0823
8,ENSG00000000460.17,C1orf112,protein_coding,703,779,726,6.6258,2.4155,3.0084
9,ENSG00000000938.13,FGR,protein_coding,2102,1068,1040,34.9716,12.7494,15.8785


## Embed with the Gen3 Inference Service

Each file is sent to the inference service's RNA-seq endpoint, which returns a 256-dimensional embedding along with quality-control stats on how many of the model's genes were found in the input.

In [ ]:
import requests
from tqdm.contrib.concurrent import thread_map

INFERENCE_API = "http://conduit-server.default.svc.cluster.local:8080/rna-seq-transformer/v1/models/bulkrnabert:predict"

def embed(file_name: str) -> dict:
    with zipfile.ZipFile(INPUT_ZIP) as zf:
        tsv = zf.read(file_name).decode()
    resp = requests.post(INFERENCE_API, json={"tsv": tsv, "sample_id": file_name})
    resp.raise_for_status()
    return resp.json()

results = thread_map(embed, metadata["file_name"], max_workers=8, desc="Embedding")

In [4]:
# results are in the same order as the metadata rows
X_input = np.asarray([r["embedding"] for r in results], dtype=np.float32)
qc = pd.DataFrame(results).drop(columns=["embedding"])

# the model's vocabulary is 19,944 genes; check how many were matched in each input
print(f"embeddings: {X_input.shape}, encoder: {', '.join(qc['encoder'].unique())}")
qc[["n_genes_vocab", "n_genes_matched", "n_genes_absent", "n_genes_unknown"]].describe()

embeddings: (20, 256), encoder: ctds-rnaseq


,n_genes_vocab,n_genes_matched,n_genes_absent,n_genes_unknown
count,20.0,20.0,20.0,20.0
mean,19944.0,19944.0,0.0,0.0
std,0.0,0.0,0.0,0.0
min,19944.0,19944.0,0.0,0.0
25%,19944.0,19944.0,0.0,0.0
50%,19944.0,19944.0,0.0,0.0
75%,19944.0,19944.0,0.0,0.0
max,19944.0,19944.0,0.0,0.0


In [5]:
# package the embeddings for download, one JSON per input file
OUTPUT_ZIP = "nciccr-dlbcl-embeddings.zip"
with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zout:
    for r in results:
        zout.writestr(r["sample_id"].removesuffix(".tsv") + ".json", json.dumps(r))

## Find the nearest TCGA neighbors

With embeddings in hand, we can use the Gen3 Embedding Service to find the 10 most similar TCGA samples to each of our samples. We search across all 32 TCGA cancer types without filtering, so any neighbors from TCGA-DLBC, which makes up less than 0.5% of TCGA, come purely from the expression embedding.

In [6]:
from gen3.auth import Gen3Auth

EMBEDDING_API = "https://genomicaicommons.org/ai/vectorstore/collections"
auth = Gen3Auth()

TOP_K = 10

def search(query_vector: np.ndarray) -> list[dict]:
    resp = requests.post(
        f"{EMBEDDING_API}/expr/search", auth=auth,
        json={"input": query_vector.tolist(), "top_k": TOP_K, "distance_metric": "cosine_similarity"},
    )
    resp.raise_for_status()
    # hits are sorted from most to least similar
    return [{"similarity": hit["value"], **hit["embedding"]["info"]["metadata"]} for hit in resp.json()["embeddings"]]

hits = thread_map(search, X_input, max_workers=8, desc="Searching")

Searching:   0%|          | 0/20 [00:00<?, ?it/s]

In [7]:
# each sample next to its nearest TCGA neighbor, plus how many of its 10 neighbors are TCGA-DLBC
nearest = pd.DataFrame([h[0] for h in hits])
pairs = pd.concat(
    {
        "NCICCR (input)": metadata[["case_id", "ann_arbor_clinical_stage", "age_in_years", "sex_at_birth"]],
        "nearest TCGA neighbor": nearest[["case_id", "project_id", "primary_site", "similarity"]],
    },
    axis=1,
)
pairs[(f"top {TOP_K} neighbors", "n TCGA-DLBC")] = [sum(x["project_id"] == "TCGA-DLBC" for x in h) for h in hits]
pairs

NCICCR (input)                                                     \
          case_id ann_arbor_clinical_stage age_in_years sex_at_birth   
0      DLBCL10451                 Stage II         73.0         male   
1      DLBCL10508                 Stage IV         64.0         male   
2      DLBCL10520                Stage III         76.0         male   
3      DLBCL10525                 Stage II         65.0         male   
4      DLBCL10861                 Stage IV         18.0       female   
5      DLBCL10877                 Stage IV         76.0       female   
6      DLBCL10880                Stage III         72.0       female   
7      DLBCL10887                 Stage IV         62.0       female   
8      DLBCL10931                 Stage IV         52.0       female   
9      DLBCL10965                Stage III         60.0         male   
10     DLBCL10973             Not Reported         66.0         male   
11     DLBCL10977                 Stage II         77.9       female   
12     DLBCL10994             Not Reported         33.0       female   
13     DLBCL11443                  Stage I         62.0         male   
14     DLBCL11445                 Stage IV         66.0         male   
15     DLBCL11487             Not Reported          0.0         male   
16     DLBCL11576                 Stage II         81.9         male   
17     DLBCL11659                  Stage I         63.0       female   
18     DLBCL11665                  Stage I         76.9       female   
19     DLBCL11676             Not Reported         68.0         male   

   nearest TCGA neighbor             \
                 case_id project_id   
0           TCGA-G8-6914  TCGA-DLBC   
1           TCGA-FF-A7CX  TCGA-DLBC   
2           TCGA-FF-8061  TCGA-DLBC   
3           TCGA-FA-A86F  TCGA-DLBC   
4           TCGA-FF-A7CR  TCGA-DLBC   
5           TCGA-FF-A7CR  TCGA-DLBC   
6           TCGA-FA-A6HO  TCGA-DLBC   
7           TCGA-FF-A7CW  TCGA-DLBC   
8           TCGA-FF-8061  TCGA-DLBC   
9           TCGA-GS-A9U4  TCGA-DLBC   
10          TCGA-FA-A6HO  TCGA-DLBC   
11          TCGA-FF-A7CR  TCGA-DLBC   
12          TCGA-FF-A7CR  TCGA-DLBC   
13          TCGA-FA-A7DS  TCGA-DLBC   
14          TCGA-FA-A7DS  TCGA-DLBC   
15          TCGA-G8-6906  TCGA-DLBC   
16          TCGA-EB-A4P0  TCGA-SKCM   
17          TCGA-FA-A6HO  TCGA-DLBC   
18          TCGA-FA-A6HO  TCGA-DLBC   
19          TCGA-FF-A7CX  TCGA-DLBC   

                                                                \
                                       primary_site similarity   
0                                       lymph nodes   0.996817   
1   connective, subcutaneous and other soft tissues   0.997924   
2                                   small intestine   0.994298   
3                                       lymph nodes   0.998970   
4                                       lymph nodes   0.995952   
5                                       lymph nodes   0.997298   
6                                       lymph nodes   0.996703   
7                                       lymph nodes   0.992355   
8                                   small intestine   0.996807   
9                                       lymph nodes   0.995636   
10                                      lymph nodes   0.997879   
11                                      lymph nodes   0.996239   
12                                      lymph nodes   0.997447   
13                                           breast   0.997181   
14                                           breast   0.998036   
15                                      lymph nodes   0.994165   
16                                             skin   0.996985   
17                                      lymph nodes   0.997473   
18                                      lymph nodes   0.994521   
19  connective, subcutaneous and other soft tissues   0.997211   

   top 10 neighbors  
        n TCGA-DLBC  
0                 8  
1                 8  
2                 

In [8]:
# which TCGA cancer types do the neighbors come from?
pd.concat(
    {
        "nearest neighbor": nearest["project_id"].value_counts(),
        f"all top {TOP_K} neighbors": pd.Series([x["project_id"] for h in hits for x in h]).value_counts(),
    },
    axis=1,
).fillna(0).astype(int)

,nearest neighbor,all top 10 neighbors
TCGA-DLBC,19,149
TCGA-SKCM,1,8
TCGA-SARC,0,13
TCGA-BLCA,0,6
TCGA-THYM,0,5
TCGA-UVM,0,4
TCGA-LIHC,0,4
TCGA-BRCA,0,3
TCGA-UCS,0,2
TCGA-ACC,0,2


## Visualize alongside all of TCGA

To see where our samples sit in the TCGA expression landscape, we pull every TCGA embedding from the Embedding Service and fit a UMAP over them together with our samples. Each line connects one of our samples to its nearest TCGA neighbor. Hover over any point to see its metadata.

In [9]:
PAGE_SIZE = 100

n_tcga = requests.get(f"{EMBEDDING_API}/expr", params={"counts": True}, auth=auth).json()["available_embeddings_count"]

def get_page(page: int) -> list[dict]:
    resp = requests.get(f"{EMBEDDING_API}/expr/embeddings", params={"page": page, "page_size": PAGE_SIZE}, auth=auth)
    resp.raise_for_status()
    return resp.json()["embeddings"]

pages = thread_map(get_page, range(1, -(-n_tcga // PAGE_SIZE) + 1), max_workers=8, desc="Downloading TCGA")
tcga = [x for page in pages for x in page]
X_tcga = np.asarray([x["vector"] for x in tcga], dtype=np.float32)
tcga = pd.DataFrame([x["info"]["metadata"] for x in tcga])
print(f"TCGA embeddings: {X_tcga.shape} across {tcga['project_id'].nunique()} projects")

TCGA embeddings: (10157, 256) across 32 projects


In [10]:
import umap

coords = umap.UMAP(metric="cosine", random_state=42, n_jobs=1).fit_transform(np.concatenate([X_tcga, X_input]))

points = pd.concat(
    [
        tcga.assign(group=np.where(tcga["project_id"] == "TCGA-DLBC", "TCGA-DLBC", "other TCGA")),
        metadata.assign(group="NCICCR-DLBCL (input)"),
    ],
    ignore_index=True,
)
points[["umap_1", "umap_2"]] = coords

In [11]:
import plotly.express as px
import plotly.graph_objects as go

fig = px.scatter(
    points, x="umap_1", y="umap_2", color="group", render_mode="webgl",
    category_orders={"group": ["other TCGA", "TCGA-DLBC", "NCICCR-DLBCL (input)"]},
    color_discrete_map={"other TCGA": "lightgray", "TCGA-DLBC": "#d62728", "NCICCR-DLBCL (input)": "#1f77b4"},
    hover_name="case_id", hover_data={"project_id": True, "primary_site": True, "group": False, "umap_1": False, "umap_2": False},
    width=900, height=700, title="NCICCR-DLBCL samples in the TCGA expression embedding space",
)

# draw a line from each input to its nearest TCGA neighbor, beneath the points
tcga_row = {fid: i for i, fid in enumerate(tcga["file_id"])}
input_xy = coords[len(X_tcga):]
nn_xy = coords[nearest["file_id"].map(tcga_row).to_numpy()]
gap = np.full(len(input_xy), np.nan)  # breaks the line between pairs
fig.add_trace(go.Scatter(
    x=np.column_stack([input_xy[:, 0], nn_xy[:, 0], gap]).ravel(),
    y=np.column_stack([input_xy[:, 1], nn_xy[:, 1], gap]).ravel(),
    mode="lines", line={"color": "black", "width": 1}, hoverinfo="skip", showlegend=False,
))
fig.data = fig.data[:1] + fig.data[-1:] + fig.data[1:-1]
fig.update_traces(selector={"name": "TCGA-DLBC"}, marker_size=7)
fig.update_traces(selector={"name": "NCICCR-DLBCL (input)"}, marker={"size": 9, "line": {"color": "white", "width": 1}})
fig.update_layout(xaxis_title="UMAP 1", yaxis_title="UMAP 2")
fig.show()